In [2]:
import os
import numpy as np
import pandas as pd

In [3]:
# don't ask what it is, I simply copied it

def create_windows(features, labels, window_size=50, step=25):

    X = []
    y = []

    for start in range(0, len(features) - window_size + 1, step):

        end = start + window_size

        window = features[start:end]
        window_labels = labels[start:end]

        # label rule (40%)
        if np.sum(window_labels) / window_size > 0.40:
            label = 1
        else:
            label = 0

        X.append(window)
        y.append(label)

    return np.array(X), np.array(y)

def extract_window_features(windows):

    feature_vectors = []

    for w in windows:

        mean = np.mean(w, axis=0)
        std = np.std(w, axis=0)
        min_val = np.min(w, axis=0)
        max_val = np.max(w, axis=0)

        features = np.concatenate([mean, std, min_val, max_val])

        feature_vectors.append(features)

    return np.array(feature_vectors)

def process_file(filepath):

    df = pd.read_csv(filepath)

    features = df.iloc[:, 2:11].values
    labels = df.iloc[:, -1].values

    windows, window_labels = create_windows(features, labels)

    window_features = extract_window_features(windows)

    return window_features, window_labels

def load_dataset(base_path):

    X_all = []
    y_all = []

    for root, dirs, files in os.walk(base_path):

        for file in files:

            if file.endswith(".csv"):

                filepath = os.path.join(root, file)

                X, y = process_file(filepath)

                X_all.append(X)
                y_all.append(y)

    X_all = np.vstack(X_all)
    y_all = np.concatenate(y_all)

    return X_all, y_all


In [4]:
train_path = "../data/Sample_Training"
test_path = "../data/Sample_Test"

X_train, y_train = load_dataset(train_path)
X_test, y_test = load_dataset(test_path)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (45664, 36)
Test shape: (10641, 36)


In [6]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200)

model.fit(X_train, y_train)

pred = model.predict(X_test)

In [ ]:
accuracy = np.mean(pred == y_test)
print("Accuracy:", accuracy)

f1_score = np.sum((pred == 1) & (y_test == 1)) / (np.sum(pred == 1) + np.sum(y_test == 1) - np.sum((pred == 1) & (y_test == 1)))
print("F1 Score:", f1_score)

Accuracy: 0.9778216333051405


In [5]:
# Random Undersampling
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler()

X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)

print("Balanced training shape:", X_train_bal.shape)

Balanced training shape: (8064, 36)


d:\Programs\Anaconda\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
d:\Programs\Anaconda\lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
d:\Programs\Anaconda\lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200)

model.fit(X_train_bal, y_train_bal)

pred = model.predict(X_test)

accuracy = np.mean(pred == y_test)
print("Accuracy:", accuracy)
f1_score = np.sum((pred == 1) & (y_test == 1)) / (np.sum(pred == 1) + np.sum(y_test == 1) - np.sum((pred == 1) & (y_test == 1)))
print("F1 Score:", f1_score)

Accuracy: 0.9258528333803214
